In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname  = "LIG",       # resname of your ligand in the topology
    topology_glob   = "*.pdb",
    trajectory_glob = "*.xtc",
    dt_ns           = 2.0,
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# Interaction types to detect (None = ProLIF defaults)
INTERACTIONS = None

OUTPUT_DIR = Path("./figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

## Step 1 — Run interaction fingerprint for all replicas

Requires: `pip install mdatools[interactions]`

In [ ]:
from mdatools.analysis.interactions import InteractionAnalyzer

analyzer = InteractionAnalyzer(cfg, interactions=INTERACTIONS)
results = analyzer.run_batch(REPLICA_ROOTS, cfg)

for res in results:
    print(f"{res.sample_name}: {len(res.df)} frames, {len(res.df.columns)} interactions")

## Step 2 — Interaction heatmap (occupancy per residue × type)

In [ ]:
from mdatools.plotting.interaction_plots import plot_interaction_heatmap

for res in results:
    fig = plot_interaction_heatmap(
        res,
        top_n=20,
        save_path=OUTPUT_DIR / f"interaction_heatmap_{res.sample_name}.png",
    )
    display(fig)

## Step 3 — Interaction timeline

In [ ]:
from mdatools.plotting.interaction_plots import plot_interaction_timeline

first = results[0]
fig = plot_interaction_timeline(
    first,
    save_path=OUTPUT_DIR / f"interaction_timeline_{first.sample_name}.png",
)
fig

## Step 4 — Comparison across replicas

In [ ]:
from mdatools.plotting.interaction_plots import plot_interaction_comparison

fig = plot_interaction_comparison(
    results,
    top_n=15,
    save_path=OUTPUT_DIR / "interaction_comparison.png",
)
fig

## Step 5 — Top interactions summary table

In [ ]:
import pandas as pd

for res in results:
    print(f"\n=== {res.sample_name} ===")
    display(res.summary.head(10).to_string(index=False))